<a id="inizio"></a>
# 07 · Pandas: operazioni sui DataFrame

<div style="background:rgba(96,125,139,0.08); border-left:4px solid #607d8b; padding:12px 16px; border-radius:4px; margin:8px 0 12px 0;">

**Blocco 3 · Analisi dei dati con pandas e visualizzazione · Giornata 2 · Aula Base &nbsp;·&nbsp; ⏱ ~90 min**

Le operazioni di tutti i giorni su un DataFrame: selezionare, pulire, trasformare, ordinare, raggruppare e unire tabelle.

**In questo notebook impariamo a**

- selezionare righe e colonne con `loc`, `iloc`, le condizioni e `.query()`
- gestire valori mancanti e duplicati, creare colonne nuove e ordinare
- raggruppare con `groupby` e unire DataFrame con `merge` e `concat`

Prima di questo: [E2 · Esercitazione 2](Esercitazione_2.ipynb) &nbsp;·&nbsp; Dati: `../Dati/U.S. Electricity Prices.csv`

</div>

<a id="indice"></a>
**Indice**

- [1. Selezione dei dati](#sez-1)
    - [1.1 Selezione di righe e colonne](#sez-1-1)
    - [1.2 Differenze tra loc e iloc](#sez-1-2)
    - [1.3 Selezione condizionale](#sez-1-3)
    - [1.4 Il metodo query](#sez-1-4)
- [2. Manipolazione dei dati](#sez-2)
    - [2.1 Valori mancanti](#sez-2-1)
    - [2.2 Duplicati](#sez-2-2)
    - [2.3 Trasformazione](#sez-2-3)
    - [2.4 Gestione degli indici](#sez-2-4)
    - [2.5 Rinomina delle colonne](#sez-2-5)
- [3. Ordinamento dei dati](#sez-3)
    - [3.1 sort_values e sort_index](#sez-3-1)
    - [3.2 Ordinamento su più colonne](#sez-3-2)
- [4. Groupby](#sez-4)
    - [4.1 Aggregazione multipla](#sez-4-1)
- [5. Unione e concatenazione di DataFrame](#sez-5)
    - [5.1 Merge](#sez-5-1)
    - [5.2 Concatenazione](#sez-5-2)
    - [5.3 Differenze tra merge e concat](#sez-5-3)
- [6. Esercizi](#sez-6)
    - [Esercizio 7.1 · Gli ascolti di un utente](#es-7-1)
    - [Esercizio 7.2 · Totale di ascolti per canzone](#es-7-2)
    - [Esercizio 7.3 · L'artista più ascoltato](#es-7-3)
    - [Esercizio 7.4 · Prezzo medio dell'elettricità per stato (facoltativo)](#es-7-4)

Importiamo le librerie e ricreiamo il DataFrame del notebook precedente: tre persone con nome,
età e città, e un indice con etichette.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# creare un DataFrame da un dizionario
data = {
    "Nome": ["Alice", "Bob", "Charlie"],
    "Età": [24, 27, 22],
    "Città": ["Roma", "Milano", "Torino"],
}
df = pd.DataFrame(data, index=["id_1", "id_2", "id_3"])
df

<a id="sez-1"></a>
## 1. Selezione dei dati

In pandas possiamo selezionare i dati per nome di colonna, per etichetta dell'indice o per
posizione.

<a id="sez-1-1"></a>
### 1.1 Selezione di righe e colonne

Possiamo selezionare una o più colonne di un DataFrame usando il nome della colonna come chiave.

In [ ]:
# selezionare una singola colonna
df[["Età"]]

In [ ]:
# selezionare più colonne
df[["Nome", "Città"]]

Con `loc` e `iloc` selezioniamo righe specifiche in base all'indice o alla posizione.

In [ ]:
# loc: in base all'indice
df.loc["id_1":"id_2", ["Età"]]

In [ ]:
# iloc: in base alla posizione
df.iloc[0:2, [1]]

<a id="sez-1-2"></a>
### 1.2 Differenze tra loc e iloc

- `loc`: seleziona i dati in base alle etichette (indice).
- `iloc`: seleziona i dati in base alla posizione intera (indice numerico).

Esempio con `loc`:

In [ ]:
df.loc["id_2":, ["Nome", "Età"]]

Esempio con `iloc`:

In [ ]:
df.iloc[1:3, :2]

`loc` permette una selezione più intuitiva quando si conoscono le etichette; con `loc` lo
slicing include anche l'ultima etichetta, con `iloc` l'ultima posizione resta esclusa.

<a id="sez-1-3"></a>
### 1.3 Selezione condizionale

Possiamo filtrare i dati con condizioni booleane. La condizione da sola restituisce una Series
di `True` e `False`, una per riga.

In [ ]:
df["Età"] > 23

Messa tra parentesi quadre, la condizione tiene solo le righe con `True`.

In [ ]:
df[df["Età"] > 23]

Più condizioni si combinano con `&` (e), `|` (o) e `~` (non), ognuna tra parentesi tonde.

In [ ]:
condiz_eta = (df["Età"] > 23) & (df["Età"] < 26)
condiz_eta

In [ ]:
# righe con età tra 23 e 26 (esclusi), solo la colonna Nome
df.loc[(df["Età"] > 23) & (df["Età"] < 26), ["Nome"]]

Queste Series di `True` e `False` si chiamano maschere booleane: sono il modo standard per
filtrare i dati in pandas.

<a id="sez-1-4"></a>
### 1.4 Il metodo query

Il metodo `.query()` filtra le righe con una condizione scritta come testo, ed è comodo da
leggere quando le condizioni sono più di una. La sintassi di base è `df.query("condizione")`.

In [ ]:
cols = ["Età", "Nome"]
df.query("Età > 23 and Età < 26").loc[:, cols]

<a id="sez-2"></a>
## 2. Manipolazione dei dati

La manipolazione dei dati in pandas comprende la gestione dei valori mancanti, la pulizia dei
dati e la trasformazione.

<a id="sez-2-1"></a>
### 2.1 Valori mancanti

I valori mancanti possono causare problemi nelle analisi e vanno gestiti. Ne esistono di tre tipi:

- `NaN` (Not a Number): usato per i valori mancanti numerici (`np.nan` di NumPy).
- `None`: l'oggetto Python che rappresenta l'assenza di valore.
- `pd.NA`: rappresenta i valori mancanti in pandas per i tipi estesi.

In [ ]:
# creare un DataFrame con valori mancanti
data_nan = {
    "A": [1, 2, np.nan],
    "B": [4, None, 6],
    "C": [7, 8, np.nan],
}
df_nan = pd.DataFrame(data_nan)
df_nan

`isnull()` (e il suo opposto `notnull()`) dice per ogni cella se il valore manca. Con `.sum()`
contiamo i mancanti per colonna.

In [ ]:
# identificare i valori mancanti
df_nan.isnull().sum()

Una colonna di interi con un `NaN` diventa di tipo `float64`. Anche `info()` mostra quanti
valori non mancanti ha ogni colonna.

In [ ]:
df_nan["A"].dtype

In [ ]:
df_nan.info()

Per risolvere i valori mancanti abbiamo tre strade:

- **Riempimento**: `fillna()` sostituisce i valori mancanti con un valore specificato.
- **Rimozione**: `dropna()` elimina le righe o le colonne con valori mancanti.
- **Interpolazione**: stima i valori mancanti basandosi sui dati esistenti.

In [ ]:
df_nan["A"].fillna(df_nan["A"].median())

In [ ]:
# riempire i valori mancanti con zero
df_nan_filled = df_nan.fillna(0)
df_nan_filled

`fillna` restituisce una copia: per modificare la colonna la riassegniamo.

In [ ]:
df_nan["A"] = df_nan["A"].fillna(df_nan["A"].median())
df_nan.dropna()

- **Quando rimuovere**: se i dati mancanti sono pochi e la rimozione non influisce sull'analisi.
- **Quando riempire**: se i dati mancanti sono molti e c'è un valore appropriato per sostituirli.

<a id="sez-2-2"></a>
### 2.2 Duplicati

I duplicati possono distorcere i risultati delle analisi e vanno identificati e gestiti.

In [ ]:
# creare un DataFrame con duplicati
data_dup = {
    "Nome": ["Alice", "Bob", "Alice"],
    "Età": [24, 27, 24],
    "Città": ["Roma", "Milano", "Roma"],
}
df_dup = pd.DataFrame(data_dup)
df_dup

In [ ]:
# identificare i duplicati
df_dup.duplicated()

In [ ]:
# rimuovere i duplicati
df_dup_clean = df_dup.drop_duplicates()
df_dup_clean

<a id="sez-2-3"></a>
### 2.3 Trasformazione

Possiamo aggiungere, modificare o eliminare colonne in un DataFrame per adattarlo alle nostre
esigenze. Una colonna nuova si crea assegnando un'espressione a un nome che non esiste ancora.

In [ ]:
df["Anni alla pensione"] = 70 - df["Età"]
df

Con `.loc` e una condizione modifichiamo solo le righe che la rispettano: qui chi ha meno di 25
anni ha il 30% di anni in più alla pensione, gli altri restano uguali.

In [ ]:
cond_riforma = df["Età"] < 25

In [ ]:
df.loc[cond_riforma, "Anni alla pensione riforma"] = df.loc[cond_riforma, "Anni alla pensione"] * 1.3
df.loc[~cond_riforma, "Anni alla pensione riforma"] = df.loc[~cond_riforma, "Anni alla pensione"]
df

<div style="background:rgba(217,83,79,0.12); border-left:4px solid #d9534f; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**⚠️ Attenzione**

`df[cond]["col"] = valore` non modifica `df`: scrive su una copia, e pandas 3 avvisa con
`ChainedAssignmentError`. Per cambiare le righe filtrate si usa sempre `df.loc[cond, "col"] = valore`.

</div>

Quando la colonna nuova non viene da un conto, `.map` traduce ogni valore con un dizionario e
`.apply` applica una funzione a ogni valore.

In [ ]:
regioni = {"Roma": "Lazio", "Milano": "Lombardia", "Torino": "Piemonte"}
df["Regione"] = df["Città"].map(regioni)
df

In [ ]:
def fascia_eta(eta):
    """Restituisce la fascia d'età di una persona."""
    if eta < 25:
        return "under 25"
    return "25 e oltre"


df["Fascia"] = df["Età"].apply(fascia_eta)
df

<a id="sez-2-4"></a>
### 2.4 Gestione degli indici

Gli indici permettono di accedere ai dati per etichetta e di allineare e unire tabelle diverse.
`set_index()` trasforma una o più colonne in indice, `reset_index()` fa il contrario.

In [ ]:
# impostare due colonne come indice
df_indexed = df.set_index(["Nome", "Città"])
df_indexed.loc[("Alice", "Roma")]

In [ ]:
# resettare l'indice
df_reset = df_indexed.reset_index()
df_reset

Dopo un ordinamento con `sort_values` (lo vediamo tra poco) le righe tengono l'indice di prima,
in disordine. `reset_index(drop=True)` lo fa ripartire da 0 e scarta il vecchio invece di
trasformarlo in una colonna.

In [ ]:
df.sort_values("Età").reset_index(drop=True)

<a id="sez-2-5"></a>
### 2.5 Rinomina delle colonne

Possiamo rinominare le colonne con `rename` e un dizionario da nome vecchio a nome nuovo.

In [ ]:
df_renamed = df.rename(columns={"Città": "Residenza", "Età": "age"})
df_renamed

<a id="sez-3"></a>
## 3. Ordinamento dei dati

L'ordinamento serve sia nell'analisi sia nella presentazione dei risultati.

<a id="sez-3-1"></a>
### 3.1 sort_values e sort_index

`sort_values()` ordina i dati in base ai valori di una o più colonne; `ascending=False` ordina
dal più grande.

In [ ]:
# ordinare per età
df_sorted = df.sort_values(by="Età", ascending=False)
df_sorted

`sort_index()` ordina in base all'indice: è utile quando l'indice ha un significato. Qui
prendiamo metà delle righe in ordine casuale con `sample` e le rimettiamo in ordine.

In [ ]:
df_shuffled = df.sample(frac=0.5)
df_shuffled.sort_index()

<a id="sez-3-2"></a>
### 3.2 Ordinamento su più colonne

Possiamo ordinare usando più colonne come chiavi: a pari valore della prima decide la seconda.
`ascending` accetta una lista, un valore per colonna.

In [ ]:
# ordinare per età e poi per nome
df_multi_sorted = df.sort_values(by=["Età", "Nome"], ascending=[False, True])
df_multi_sorted

<a id="sez-4"></a>
## 4. Groupby

Il metodo `groupby()` **suddivide** un DataFrame in gruppi in base a una o più colonne, per poi
applicare un'**aggregazione** ai dati di ciascun gruppo. È la logica **Split-Apply-Combine**:

1. **Split**: dividere i dati in gruppi in base a una o più colonne.
2. **Apply**: applicare una funzione (somma, media, conteggio) a ogni gruppo.
3. **Combine**: unire i risultati in un nuovo oggetto pandas (Series o DataFrame).

La forma è `df.groupby("colonna_di_raggruppamento")["colonna"].funzione()`.

In [ ]:
vendite = pd.DataFrame({
    "Categoria": ["A", "B", "A", "B", "A", "C"],
    "Vendite": [100, 200, 150, 250, 120, 300],
})

# raggruppiamo per Categoria e sommiamo le vendite
vendite.groupby("Categoria")["Vendite"].sum()

`groupby()` da solo restituisce un oggetto GroupBy, che aspetta una funzione per elaborare i
gruppi. Con `sum()` otteniamo una riga per gruppo, con la categoria nell'indice.

<a id="sez-4-1"></a>
### 4.1 Aggregazione multipla

Con `.agg()` calcoliamo più statistiche in una volta sola.

In [ ]:
vendite.groupby("Categoria").agg({"Vendite": ["sum", "mean", "count"]})["Vendite"]

<a id="sez-5"></a>
## 5. Unione e concatenazione di DataFrame

Pandas offre diversi modi per combinare più DataFrame.

<a id="sez-5-1"></a>
### 5.1 Merge

`merge()` combina DataFrame in base a chiavi comuni, come un join in SQL. Tipi di join:

- **Inner join**: le righe con chiavi presenti in entrambi i DataFrame.
- **Left join**: tutte le righe del DataFrame di sinistra e quelle corrispondenti di destra.
- **Right join**: tutte le righe del DataFrame di destra e quelle corrispondenti di sinistra.
- **Outer join**: tutte le righe, che abbiano una corrispondenza o no.

In [ ]:
# DataFrame di esempio
df_left = pd.DataFrame({"ID": [1, 2, 3], "Nome": ["Alice", "Bob", "Charlie"]})
df_right = pd.DataFrame({"ID": [3, 4, 5], "Età": [35, 40, 45]})

# inner join
df_inner = pd.merge(df_left, df_right, on="ID", how="inner")
df_inner

In [ ]:
# outer join
df_outer = pd.merge(df_left, df_right, on="ID", how="outer")
df_outer

Con il left join teniamo tutte le righe del DataFrame di sinistra: dove manca la corrispondenza
resta `NaN`.

In [ ]:
# left join
pd.merge(df_left, df_right, on="ID", how="left")

<a id="sez-5-2"></a>
### 5.2 Concatenazione

`concat()` unisce DataFrame verticalmente (aggiunge righe) o orizzontalmente (aggiunge colonne).

In [ ]:
# concatenazione verticale
df_concat_vertical = pd.concat([df_left, df_right], ignore_index=True, sort=False)
df_concat_vertical

In [ ]:
# concatenazione orizzontale
df_concat_horizontal = pd.concat([df_left, df_right], axis=1)
df_concat_horizontal

<a id="sez-5-3"></a>
### 5.3 Differenze tra merge e concat

- **Merge**: quando vogliamo combinare DataFrame su una o più chiavi comuni.
- **Concat**: quando vogliamo unire DataFrame che hanno le stesse colonne o lo stesso indice.

<a id="sez-6"></a>
## 6. Esercizi

Lavori come data analyst per un'azienda che gestisce una piattaforma di streaming musicale.
Il dataset con gli ascolti degli utenti è questo: eseguilo prima dei primi tre esercizi.

In [ ]:
# DataFrame degli ascolti
listens_data = {
    "UserID": [101, 102, 103, 104, 105, 106],
    "Song": ["Song A", "Song B", "Song A", "Song C", "Song B", "Song D"],
    "Artist": ["Artist X", "Artist Y", "Artist X", "Artist Z", "Artist Y", "Artist W"],
    "Plays": [15, 2, 4, 1, 5, 2],
}
listens = pd.DataFrame(listens_data)
listens

<a id="es-7-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 7.1 · Gli ascolti di un utente**



**Cosa fare.**

Trova tutte le canzoni ascoltate dall'utente con `UserID` 103: filtra `listens` per tenere
solo le righe dove `UserID` è 103 e salva il risultato in `user_103_listens`.

Output atteso: una riga, `Song A` di `Artist X` con 4 ascolti.

*Suggerimento:* usa il filtraggio condizionale con una maschera booleana.

</div>

In [ ]:
cond = ...
user_103_listens = ...
user_103_listens

In [ ]:
# Verifica: esegui senza modificare
assert len(user_103_listens) == 1, "❌ Deve restare una sola riga"
assert user_103_listens["Song"].tolist() == ["Song A"], "❌ La canzone dell'utente 103 è Song A"
assert user_103_listens["Plays"].tolist() == [4], "❌ Gli ascolti dell'utente 103 sono 4"
print("✅ Esercizio 7.1 completato")

<a id="es-7-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 7.2 · Totale di ascolti per canzone**



**Cosa fare.**

Calcola il totale di ascolti per ogni canzone: raggruppa per `Song` e somma i valori di
`Plays`. Salva il risultato in `total_plays_per_song`.

Output atteso: Song A 19, Song B 7, Song C 1, Song D 2.

*Suggerimento:* usa `groupby()` seguito da `sum()`.

</div>

In [ ]:
total_plays_per_song = listens.groupby("...")["..."].sum()
total_plays_per_song

In [ ]:
# Verifica: esegui senza modificare
expected = {"Song A": 19, "Song B": 7, "Song C": 1, "Song D": 2}
assert total_plays_per_song.to_dict() == expected, "❌ Raggruppa per Song e somma Plays"
print("✅ Esercizio 7.2 completato")

<a id="es-7-3"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 7.3 · L'artista più ascoltato**



**Cosa fare.**

Identifica l'artista più popolare: raggruppa per `Artist`, somma `Plays` e ordina in modo
decrescente (`artist_popularity`). Poi salva in `top_artist` il nome dell'artista con il
totale di ascolti più alto.

*Suggerimento:* combina `groupby()`, `sum()` e `sort_values()`; `idxmax()` restituisce l'etichetta del valore massimo.

</div>

In [ ]:
artist_popularity = listens.groupby(...)[...].sum().sort_values(ascending=False)
top_artist = artist_popularity.idxmax()
top_artist

In [ ]:
# Verifica: esegui senza modificare
assert artist_popularity.iloc[0] == 19, "❌ Il primo totale, in ordine decrescente, è 19"
assert top_artist == "Artist X", "❌ L'artista più ascoltato è un altro"
print("✅ Esercizio 7.3 completato")

<a id="es-7-4"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 7.4 · Prezzo medio dell'elettricità per stato (facoltativo)**

Il file `U.S. Electricity Prices.csv` contiene il prezzo medio mensile dell'elettricità negli
Stati Uniti (in centesimi di dollaro per kWh), per stato e per settore, dal 2001 al 2024.

**Cosa fare.**

1. Leggi il file in `prezzi` e tieni solo le righe con `sectorName` uguale a `"all sectors"`.
2. Calcola il prezzo medio (`price`) per stato (`stateDescription`) in `prezzo_medio`,
   con `reset_index()` per riavere lo stato come colonna.
3. Unisci `stati` (già definito nella cella) e `prezzo_medio` con un inner join sulla
   colonna `stateDescription`, nel DataFrame `prezzi_stati`.

Output atteso: quattro righe; il Texas ha un prezzo medio di circa 8,86.

*Suggerimento:* guarda prima le colonne con `prezzi.head()`.

</div>

In [ ]:
prezzi = pd.read_csv("../Dati/U.S. Electricity Prices.csv")
stati = pd.DataFrame({
    "stateDescription": ["California", "Texas", "New York", "Florida"],
    "sigla": ["CA", "TX", "NY", "FL"],
})

tutti_settori = prezzi[...]
prezzo_medio = tutti_settori.groupby(...)[...].mean().reset_index()
prezzi_stati = pd.merge(..., ..., on=..., how="inner")
prezzi_stati

In [ ]:
# Verifica: esegui senza modificare
assert len(prezzi_stati) == 4, "❌ Con l'inner join restano le quattro righe di stati"
assert {"sigla", "price"} <= set(prezzi_stati.columns), "❌ Servono le colonne sigla e price"
prezzo_tx = prezzi_stati.loc[prezzi_stati["sigla"] == "TX", "price"].iloc[0]
assert round(prezzo_tx, 2) == 8.86, "❌ Filtra su all sectors prima della media"
print("✅ Esercizio 7.4 completato")

---

**Fine del notebook 07.** Prossimo: [08 · Pandas: le date](08_Pandas_date.ipynb)